In [ ]:
import pandas as pd
from tqdm import tqdm
import numpy as np
import tensorflow as tf
!pip install ipywidgets
!pip install -U accelerate
!pip install -U transformers
!pip install transformers[torch]
!pip install accelerate -U
!pip install sacrebleu
!pip install evaluate
import transformers
import accelerate

***

# -- Fine Tuning Pre-trained model --

In [ ]:
data = pd.read_csv("/kaggle/input/hindienglish-corpora/Hindi_English_Truncated_Corpus.csv")
data.drop(columns=["source"],inplace=True)
data.dropna(inplace=True)

num_epochs = 10

In [ ]:
from datasets import Dataset
data = Dataset.from_pandas( data )
data = data.train_test_split(test_size=0.15)
data

In [ ]:
data["train"][0]

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "Helsinki-NLP/opus-mt-en-hi"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, return_tensors="pt")

In [ ]:
inputs = tokenizer(data["train"][0]['english_sentence'], text_target=data["train"][0]['hindi_sentence'])
inputs

In [ ]:
max_length = 128

def preprocess_function(data):
    inputs = [ex for ex in data["english_sentence"]]
    targets = [ex for ex in data["hindi_sentence"]]
    model_inputs = tokenizer( inputs, text_target=targets, max_length=max_length, truncation=True )
    return model_inputs

In [ ]:
tokenized_datasets = data.map(preprocess_function,batched=True, remove_columns=data["train"].column_names)

In [ ]:
from transformers import TFAutoModelForSeq2SeqLM

model = TFAutoModelForSeq2SeqLM.from_pretrained(model_checkpoint, from_pt=True)

In [ ]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, return_tensors="tf", pad_to_multiple_of=128)

In [ ]:
batch_size = 16
tf_train_dataset = model.prepare_tf_dataset( tokenized_datasets["train"], collate_fn=data_collator, shuffle=True, batch_size=batch_size )
tf_eval_dataset = model.prepare_tf_dataset( tokenized_datasets["test"], collate_fn=data_collator, shuffle=False, batch_size=batch_size )

In [ ]:
import transformers
import accelerate
import evaluate
metric = evaluate.load("sacrebleu")

In [ ]:
@tf.function(jit_compile=True)
def generate_with_xla(batch):
    return model.generate( input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], max_new_tokens=128 )

def compute_metrics():
    all_preds = []
    all_labels = []
    for batch, labels in tqdm(tf_eval_dataset):
        predictions = generate_with_xla(batch)
        decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
        labels = labels.numpy()
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        decoded_preds = [pred.strip() for pred in decoded_preds]
        decoded_labels = [[label.strip()] for label in decoded_labels]
        all_preds.extend(decoded_preds)
        all_labels.extend(decoded_labels)

    result = metric.compute(predictions=all_preds, references=all_labels)
    return {"bleu": result["score"]}

In [ ]:
from huggingface_hub import notebook_login, login
# notebook_login()
login(token="YOUR_HF_TOKEN", write_permission=True)

In [ ]:
print(compute_metrics())

In [ ]:
from transformers import create_optimizer
from transformers.keras_callbacks import PushToHubCallback
import tensorflow as tf

num_train_steps = len(tf_train_dataset) * num_epochs

optimizer, schedule = create_optimizer(init_lr=5e-5, num_warmup_steps=0, num_train_steps=num_epochs, weight_decay_rate=0.01 )
model.compile(optimizer=optimizer)

In [ ]:
from transformers.keras_callbacks import PushToHubCallback

callback = PushToHubCallback( output_dir="Helsinki-shirsh-finetuned-translation-english-to-hindi", tokenizer=tokenizer )

model.fit( tf_train_dataset, validation_data=tf_eval_dataset, callbacks=[callback], epochs=num_epochs )

print(compute_metrics())

In [ ]:
from transformers import pipeline

# Replace this with your own checkpoint
model_checkpoint = "shirsh10mall/Helsinki-shirsh-finetuned-translation-english-to-hindi"
translator = pipeline("translation", model=model_checkpoint)

In [ ]:
translator("Hello")

In [ ]:
len(data['train']["english_sentence"])

In [ ]:
print( "-- Training Set -- " )
for i in range(10):
    index = np.random.randint(len(data['train']["english_sentence"]),size=1)
    print("English Sentence: ", data['train']["english_sentence"][index[0]])
    print("Original Hindi Sentence: ", data['train']["hindi_sentence"][index[0]])
    print("Translated Hindi Sentence: ", translator( data['train']["english_sentence"][index[0]])[0]["translation_text"])
    print('\n')

In [ ]:
print( " -- Validation/Testing Set -- " )
for i in range(10):
    index = np.random.randint(len(data['test']["english_sentence"]),size=1)
    print("English Sentence: ", data['test']["english_sentence"][index[0]])
    print("Original Hindi Sentence: ", data['test']["hindi_sentence"][index[0]])
    print("Translated Hindi Sentence: ", translator( data['test']["english_sentence"][index[0]])[0]["translation_text"])
    print('\n')

***